In [ ]:
!pip install -q torch-geometric==2.6.1 biopython scipy
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y default-jre-headless > /dev/null 2>&1; java -version 2>&1 | head -1

In [ ]:
"""Phase 227 -- hand-crafted per-residue geometry features, the control arm for the TDA claim.

Question: does topology add anything beyond the burial / shape / exposure descriptors a structural
biologist would compute with no TDA at all? Phase 164 already showed a 2-feature burial baseline
(heavy-atom count within 12 A + gyration) recovers most of the TDA gain, so the control here is a
much richer, deliberately strong plain-geometry set.

All features are functions of the same input the TDA block sees: heavy atoms of the standard residues
of ONE chain, first model, residue filter identical to the ingestion (resname in AA3, has CA).
Every feature is invariant to rigid motion except for Shrake-Rupley sampling noise in rel_sasa and
residue_depth (lab-frame sphere points); that noise is measured in the test, not assumed away.
"""
import gzip
import numpy as np
from scipy.spatial import cKDTree, ConvexHull
from Bio.PDB import MMCIFParser, PDBParser
from Bio.PDB.SASA import ShrakeRupley

AA3 = {"MSE": "M", "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q", "GLU": "E", "GLY": "G",
       "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P", "SER": "S",
       "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V"}
# Tien et al. 2013 theoretical maximum residue ASA (A^2), used to make SASA relative
MAX_ASA = {"ALA": 129, "ARG": 274, "ASN": 195, "ASP": 193, "CYS": 167, "GLN": 225, "GLU": 223, "GLY": 104,
           "HIS": 224, "ILE": 197, "LEU": 201, "LYS": 236, "MET": 224, "PHE": 240, "PRO": 159, "SER": 155,
           "THR": 172, "TRP": 285, "TYR": 263, "VAL": 174}
COUNT_RADII = (4.0, 6.0, 8.0, 10.0, 12.0, 15.0)
HSE_RADIUS = 13.0
SHAPE_RADIUS = 12.0
N_RAYS = 100
RAY_LENGTHS = (10.0, 20.0)
RAY_STEP = 1.0
RAY_START = 2.0
SASA_POINTS = 200
EXPOSED_ATOM_MIN_SASA = 5.0   # A^2; an atom counts as solvent exposed above this
RAY_CLEARANCE = 1.7          # a ray sample within this distance of a heavy atom counts as blocked
NAMES = ([f"n_atoms_{int(r)}A" for r in COUNT_RADII]
         + ["rel_sasa", "hse_up", "hse_down", "shape_linear", "shape_planar", "shape_spherical",
            "offset_from_local_centroid", "ray_blocked_10A", "ray_blocked_20A", "hull_depth",
            "residue_depth"])
GEOM_DIM = len(NAMES)


def _fib_sphere(n):
    i = np.arange(n) + 0.5
    phi = np.arccos(1 - 2 * i / n)
    th = np.pi * (1 + 5 ** 0.5) * i
    return np.stack([np.cos(th) * np.sin(phi), np.sin(th) * np.sin(phi), np.cos(phi)], 1)


RAYS = _fib_sphere(N_RAYS)


def _local_frame(pts, c):
    """Orthonormal frame (rows = axes) from the local atom cloud; sign of each axis from the skew of the
    projections so a rigid rotation of the input rotates the frame with it."""
    if len(pts) < 4:
        return np.eye(3)
    q = pts - pts.mean(0)
    w, v = np.linalg.eigh(np.cov(q.T))
    v = v[:, ::-1].copy()
    for k in range(3):
        if (q @ v[:, k]).__pow__(3).sum() < 0:
            v[:, k] = -v[:, k]
    if np.linalg.det(v) < 0:
        v[:, 2] = -v[:, 2]
    return v


def load_chain(path, chain_id, pid="x"):
    """Same filter as the ingestion: standard residues with a CA, first model, one chain, no hetero."""
    op = gzip.open if str(path).endswith(".gz") else open
    parser = MMCIFParser(QUIET=True) if ".cif" in str(path) else PDBParser(QUIET=True, PERMISSIVE=True)
    with op(path, "rt") as fh:
        model = next(parser.get_structure(pid, fh).get_models())
    chain = model[chain_id]
    for r in list(chain):
        # same residue filter as the ingestion: resname in AA3_TO_1 (which includes selenomethionine,
        # a HETATM record) and a CA atom -- no hetero-flag test, or MSE residues would be dropped
        if r.get_resname().strip().upper() not in AA3 or "CA" not in r:
            chain.detach_child(r.id)
    for r in chain:
        for a in list(r):
            if (a.element or "").strip().upper() in ("H", "D"):
                r.detach_child(a.id)
    return chain


def geometry_for_chain(chain):
    residues = list(chain)
    n = len(residues)
    ca = np.array([r["CA"].coord for r in residues], float)
    atoms, owner = [], []
    for i, r in enumerate(residues):
        for a in r:
            atoms.append(a.coord)
            owner.append(i)
    atoms = np.asarray(atoms, float)
    tree = cKDTree(atoms)

    # SASA (atom level, then residue sum) on this chain alone
    ShrakeRupley(n_points=SASA_POINTS).compute(chain, level="A")
    atom_sasa = np.array([a.sasa for r in residues for a in r], float)
    res_sasa = np.bincount(owner, weights=atom_sasa, minlength=n)
    rel_sasa = np.array([min(res_sasa[i] / MAX_ASA["MET" if residues[i].get_resname().strip().upper() == "MSE" else residues[i].get_resname().strip().upper()], 1.5)
                         for i in range(n)])
    exposed_tree = cKDTree(atoms[atom_sasa > EXPOSED_ATOM_MIN_SASA]) if (atom_sasa > EXPOSED_ATOM_MIN_SASA).any() else None

    out = np.zeros((n, GEOM_DIM), np.float32)
    ca_tree = cKDTree(ca)
    hull = None
    try:
        hull = ConvexHull(atoms)
    except Exception:
        pass

    for i in range(n):
        c = ca[i]
        col = 0
        for r in COUNT_RADII:
            out[i, col] = len(tree.query_ball_point(c, r)); col += 1
        out[i, col] = rel_sasa[i]; col += 1

        # half-sphere exposure (CA-based): direction bisecting the two CA-CA vectors
        prev_v = ca[i - 1] - c if i > 0 else None
        next_v = ca[i + 1] - c if i < n - 1 else None
        vs = [v for v in (prev_v, next_v) if v is not None]
        u = np.sum(vs, axis=0) if vs else np.zeros(3)
        nu = np.linalg.norm(u)
        u = u / nu if nu > 1e-6 else u
        nb = [j for j in ca_tree.query_ball_point(c, HSE_RADIUS) if j != i]
        d = ca[nb] - c if nb else np.zeros((0, 3))
        proj = d @ u if len(nb) else np.zeros(0)
        out[i, col] = float((proj > 0).sum()); out[i, col + 1] = float((proj <= 0).sum()); col += 2

        # local shape of the heavy-atom cloud
        idx = tree.query_ball_point(c, SHAPE_RADIUS)
        pts = atoms[idx]
        if len(pts) >= 4:
            cen = pts.mean(0)
            ev = np.sort(np.linalg.eigvalsh(np.cov((pts - cen).T)))[::-1]
            ev = np.maximum(ev, 0.0)
            l1 = max(ev[0], 1e-9)
            out[i, col:col + 3] = [(ev[0] - ev[1]) / l1, (ev[1] - ev[2]) / l1, ev[2] / l1]
            out[i, col + 3] = float(np.linalg.norm(c - cen))
        col += 4

        # ray buriedness, with rays expressed in a canonical local frame (PCA axes of the local atom
        # cloud, signs fixed by the third moment) so the feature is rotation invariant
        R = _local_frame(pts, c)
        rays = RAYS @ R.T
        ts = np.arange(RAY_START, max(RAY_LENGTHS) + 1e-9, RAY_STEP)
        samples = c[None, None, :] + rays[:, None, :] * ts[None, :, None]
        dist, _ = tree.query(samples.reshape(-1, 3), distance_upper_bound=RAY_CLEARANCE)
        hit = np.isfinite(dist).reshape(N_RAYS, len(ts))
        for L in RAY_LENGTHS:
            out[i, col] = float(hit[:, ts <= L + 1e-9].any(1).mean()); col += 1

        # depth below the convex hull surface, and residue depth (mean distance to the 3 nearest exposed atoms)
        if hull is not None:
            out[i, col] = float(np.min(-(hull.equations[:, :3] @ c + hull.equations[:, 3])))
        col += 1
        if exposed_tree is not None:
            dd = exposed_tree.query(c, k=min(3, exposed_tree.n))[0]
            out[i, col] = float(np.mean(dd))
        col += 1
    return out, ca

In [ ]:
import os, re, time, subprocess, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, torch
from joblib import Parallel, delayed

def locate(f):
    hits = []
    for root in (Path('/kaggle/input'), Path('/kaggle/working'), Path('.')):
        if root.exists(): hits.extend(root.rglob(f))
    if not hits: raise FileNotFoundError(f)
    return max(hits, key=lambda p: p.stat().st_size)

blob = torch.load(locate('phase48_cryptobench_graphs_esm3b.pt'), map_location='cpu', weights_only=False)
graphs = {}
for d in blob['dataset']:
    if str(d.split) in ('val', 'test'):
        graphs[(str(d.pdb_id).lower(), str(d.chain_id))] = dict(n=int(d.num_nodes), pos=d.pos.numpy(), split=str(d.split), tc=d.true_center.numpy().reshape(-1), y=d.y.numpy().astype(np.int8))
del blob
print(len(graphs), 'val+test chains:', {s: sum(g['split'] == s for g in graphs.values()) for s in ('val', 'test')})

WORK = Path('/kaggle/working'); CIF = WORK / 'apo_cifs'; CIF.mkdir(exist_ok=True)
def fetch(pid):
    dst = CIF / f'{pid}.cif'
    if dst.exists() and dst.stat().st_size > 1000: return pid, True
    for attempt in range(4):
        try:
            with urllib.request.urlopen(f'https://files.rcsb.org/download/{pid.upper()}.cif', timeout=60) as r: dst.write_bytes(r.read())
            return pid, True
        except Exception: time.sleep(2 * (attempt + 1))
    return pid, False
pids = sorted({k[0] for k in graphs})
res = Parallel(n_jobs=8, backend='threading')(delayed(fetch)(p) for p in pids)
missing = [p for p, ok in res if not ok]; print(f'fetched {len(pids)-len(missing)}/{len(pids)}; missing {missing}'); assert not missing

# verify alignment + record residue ids in node order
info = {}
for key, g in sorted(graphs.items()):
    try:
        ch = load_chain(CIF / f'{key[0]}.cif', key[1], key[0]); rs = list(ch)
        ca = np.array([r['CA'].coord for r in rs], float)
        assert len(rs) == g['n'], f"node count {len(rs)} vs {g['n']}"
        err = float(np.abs((ca - ca.mean(0)) - g['pos']).max()); assert err < 1e-3, f'CA mismatch {err:.4f}'
        info[key] = dict(shift=ca.mean(0), ids=[(r.id[1], (r.id[2] or '').strip()) for r in rs])
    except Exception as e:
        print('ALIGN FAIL', key, type(e).__name__, e)
print(f'aligned {len(info)}/{len(graphs)} chains')
assert len(info) >= 0.95 * len(graphs)

In [ ]:
subprocess.run("curl -sL https://github.com/rdk/p2rank/releases/download/2.5.1/p2rank_2.5.1.tar.gz -o /tmp/p2.tgz && tar xzf /tmp/p2.tgz -C /tmp", shell=True, check=True, timeout=1200)
prank = next(Path('/tmp').glob('p2rank_*/prank')); os.chmod(prank, 0o755)
ds = WORK / 'val_test.ds'
ds.write_text('HEADER: protein\n' + ''.join(f'{CIF}/{p}.cif\n' for p in sorted({k[0] for k in info})))
OUT = WORK / 'p2rank_out'
t0 = time.time(); r = subprocess.run([str(prank), 'predict', str(ds), '-o', str(OUT), '-threads', str(os.cpu_count() or 2)], capture_output=True, text=True, timeout=10800)
print('prank rc', r.returncode, f'{(time.time()-t0)/60:.1f} min'); print(r.stdout[-600:]); print(r.stderr[-600:])

In [ ]:
label_re = re.compile(r"^(-?\d+)([A-Za-z]?)$")
out = {}; fails = []
for key, inf in info.items():
    pdb, chain = key; g = graphs[key]
    try:
        rd = pd.read_csv(OUT / f'{pdb}.cif_residues.csv', skipinitialspace=True); rd.columns = [c.strip() for c in rd.columns]
        rd['chain'] = rd['chain'].astype(str).str.strip(); rd = rd[rd.chain == chain]
        lab = rd['residue_label'].astype(str).str.strip().str.extract(label_re)
        rd = rd.assign(resnum=pd.to_numeric(lab[0]), icode=lab[1].fillna(''))
        idx = {(int(a), b): i for i, (a, b) in enumerate(zip(rd.resnum, rd.icode))}
        rows = [idx.get((int(a), b)) for a, b in inf['ids']]
        miss = sum(x is None for x in rows)
        arr = np.full((g['n'], 3), np.nan, np.float32)
        for i, x in enumerate(rows):
            if x is not None: arr[i] = rd.iloc[x][['score', 'zscore', 'probability']].astype(float).values
        pk = pd.read_csv(OUT / f'{pdb}.cif_predictions.csv', skipinitialspace=True); pk.columns = [c.strip() for c in pk.columns]
        pockets = []
        for _, q in pk.iterrows():
            ids = str(q['residue_ids']).split(); frac = np.mean([i.split('_')[0] == chain for i in ids]) if ids else 0.0
            pockets.append(dict(rank=int(q['rank']), score=float(q['score']), prob=float(q['probability']), frac_on_chain=float(frac),
                                center=np.array([q['center_x'], q['center_y'], q['center_z']], float) - inf['shift']))
        out[key] = dict(split=g['split'], pos=g['pos'], tc=g['tc'], y=g['y'], p2=arr, n_unmatched=miss, pockets=pockets)
    except Exception as e:
        fails.append((key, f'{type(e).__name__}: {e}'))
print(f'P2Rank aligned for {len(out)}/{len(info)} chains; failures {len(fails)}; chains with unmatched residues {sum(v["n_unmatched"]>0 for v in out.values())}; total unmatched nodes {sum(v["n_unmatched"] for v in out.values())} of {sum(v["p2"].shape[0] for v in out.values())}')
for f in fails[:20]: print('  FAIL', f)
assert len(out) >= 0.9 * len(graphs)
torch.save(out, WORK / 'phase267_p2rank_and_coords.pt')
import shutil; shutil.rmtree(CIF, ignore_errors=True); shutil.rmtree(OUT, ignore_errors=True)